In [ ]:
import xarray as xr
import numpy as np
import tqdm
from scipy.stats import pearsonr
from skimage.measure import block_reduce
from src.configuration import Config
import src.utils.xarray_utils as xu

/opt/conda/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [5]:
# load datasets
scratch_path = '/dss/dssfs04/lwp-dss-0002/pn49fu/pn49fu-dss-0002/ge45tac2/diffusion-downscaling/'

config = Config(out_path=scratch_path)

test_split = slice('2004', '2018')
train_split = slice('1950', '1990')

config.target_filename = 'era5_4x_downscaled_v3.nc'
era5_train = xr.open_dataset(config.data_path + '/' + config.target_filename, chunks={'time': 1}).precipitation.sel(time=train_split)*(24*3600)
era5 = xr.open_dataset(config.data_path + '/' + config.target_filename).precipitation.sel(time=test_split)*(24*3600)

config.esm_filename = 'poem_historical_3deg_4xdown_bilinear_lowpass_dqm.nc'
esm_hr_train = xr.open_dataset(config.data_path + '/' + config.esm_filename, chunks={'time': 1}).precipitation.sel(time=train_split)*(24*3600)
esm_hr = xr.open_dataset(config.data_path + '/' + config.esm_filename).precipitation.sel(time=test_split)*(24*3600)
esm_hr = xu.shift_longitudes(esm_hr)

config.esm_filename = 'poem_historical_3deg_4xdown_bilinear_v3.nc'
esm_hr_no_qm = xr.open_dataset(config.data_path + '/' + config.esm_filename).precipitation.sel(time=test_split)*(24*3600)
esm_hr_no_qm = xu.shift_longitudes(esm_hr_no_qm)

esm_lr_filename = 'poem_historical_3deg.nc'
esm_lr = xr.open_dataset(config.data_path + '/' + esm_lr_filename, chunks={'time': 1}).precipitation.sel(time=test_split)*(24*3600)
esm_lr = xu.shift_longitudes(esm_lr)

sde_filename = scratch_path+'results/sde_historical_poem_4x_dqm_2004_2018.nc'
sde = xr.open_dataset(sde_filename, chunks={'time': 1}).precipitation.sel(time=test_split)*(24*3600)
sde = xu.shift_longitudes(sde)

cm_filename = scratch_path+'results/cm_historical_scale_468.nc'
cm = xr.open_dataset(cm_filename, chunks={'time': 1}).precipitation*(24*3600)
cm = xu.shift_longitudes(cm)

In [6]:
# helper functions
def pool(x, stride):
    return block_reduce(x, (stride,stride), np.mean)

def correlation(a, b):
    return pearsonr(a.flatten(), b.flatten()).statistic

# Table

In [65]:
num_t = len(sde)
sde_corr_pooled = np.zeros(num_t)
cm_corr_pooled = np.zeros(num_t)

for t in tqdm.tqdm(range(num_t)):
    esm_slice = esm_hr.isel(time=t)
    esm_pooled = pool(esm_slice.values, stride=4)

    sde_slice = sde.isel(time=t)
    sde_pooled = pool(sde_slice.values, stride=4)
    sde_corr_pooled[t] = correlation(sde_pooled, esm_pooled)
    
    cm_slice = cm.isel(time=t)
    cm_pooled = pool(cm_slice.values, stride=4)
    cm_corr_pooled[t] = correlation(cm_pooled, esm_pooled)

100%|██████████| 5475/5475 [01:57<00:00, 46.63it/s]


In [73]:
print(f"SDE correlation: mean = {sde_corr_pooled.mean():2.3f}, std = {sde_corr_pooled.std():2.3f}")
print(f"CM correlation: mean = {cm_corr_pooled.mean():2.3f}, std = {cm_corr_pooled.std():2.3f}")

SDE correlation: mean = 0.918, std = 0.014
CM correlation: mean = 0.954, std = 0.008


In [73]:
from src.utils import compute_filtered_correlations, low_pass_filter

sde_lp = low_pass_filter(sde, cutoff=0.0667)
cm_lp = low_pass_filter(cm, cutoff=0.0667)

num_t = len(sde)
sde_corr_filtered = np.zeros(num_t)
cm_corr_filtered = np.zeros(num_t)

for t in tqdm.tqdm(range(num_t)):
    esm_slice = esm_hr[t].values
    
    sde_slice = sde_lp[t]
    sde_corr_filtered[t] = correlation(sde_slice, esm_slice)
    
    cm_slice = cm_lp[t]
    cm_corr_filtered[t] = correlation(cm_slice, esm_slice)

100%|██████████| 5475/5475 [00:41<00:00, 131.85it/s]


In [74]:
print(f"SDE correlation: mean = {sde_corr_filtered.mean():2.3f}, std = {sde_corr_filtered.std():2.3f}")
print(f"CM correlation: mean = {cm_corr_filtered.mean():2.3f}, std = {cm_corr_filtered.std():2.3f}")

SDE correlation: mean = 0.916, std = 0.012
CM correlation: mean = 0.941, std = 0.009


In [14]:
esm_hr_no_qm['time'] =  cm.time 
cm_mae = abs(era5.mean(dim="time") - cm.mean(dim="time")).mean().values
sde_mae = abs(era5.mean(dim="time") - sde.mean(dim="time")).mean().values
esm_mae = abs(era5.mean(dim="time") - esm_hr_no_qm.mean(dim="time")).mean().values

print(f"SDE MAE = {sde_mae:2.3f}")
print(f"CM MAE = {cm_mae:2.3f}")
print(f"ESM MAE = {esm_mae:2.3f}")

SDE MAE = 0.214
CM MAE = 0.217
ESM MAE = 0.778


In [19]:
def relative_error_reduction(error_1, error_2):
    return (error_1 - error_2) / error_2 * 100.0
    
print(f"SDE rel. error reduction = {relative_error_reduction(sde_mae, esm_mae):2.3f}")
print(f"CM rel. error reduction = {relative_error_reduction(cm_mae, esm_mae):2.3f}")

SDE rel. error reduction = -72.506
CM rel. error reduction = -72.081


In [29]:
esm_hr_no_qm['time'] =  cm.time 
q = 0.95
cm_mae = abs(era5.load().quantile(q, dim="time") - cm.load().quantile(q, dim="time")).mean().values
sde_mae = abs(era5.load().quantile(q, dim="time") - sde.load().quantile(q, dim="time")).mean().values
esm_mae = abs(era5.load().quantile(q, dim="time") - esm_hr_no_qm.load().quantile(q, dim="time")).mean().values

print(f"SDE MAE = {sde_mae:2.3f}")
print(f"CM MAE = {cm_mae:2.3f}")
print(f"ESM MAE = {esm_mae:2.3f}")

SDE MAE = 1.106
CM MAE = 1.080
ESM MAE = 3.474


In [30]:
def relative_error_reduction(error_1, error_2):
    return (error_1 - error_2) / error_2 * 100.0
    
print(f"SDE rel. error reduction = {relative_error_reduction(sde_mae, esm_mae):2.3f}")
print(f"CM rel. error reduction = {relative_error_reduction(cm_mae, esm_mae):2.3f}")

SDE rel. error reduction = -68.154
CM rel. error reduction = -68.920
